In [1]:
import torch
import pandas as pd
import sklearn

print("torch:", torch.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("device:", "mps" if torch.backends.mps.is_available() else "cpu")

torch: 2.11.0
pandas: 3.0.1
sklearn: 1.8.0
device: mps


In [2]:
import torch

# 创建 tensor
x = torch.tensor([2.0, 3.0, 4.0])
print("shape:", x.shape)
print("dtype:", x.dtype)
print("x * 2 =", x * 2)
print("mean:", x.mean().item())
print("sum:", x.sum().item())

shape: torch.Size([3])
dtype: torch.float32
x * 2 = tensor([4., 6., 8.])
mean: 3.0
sum: 9.0


In [6]:
# 定义两个可学习参数
w = torch.tensor(1.5, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# 假设输入是 3.0，真实值是 6.0
x_input = 3.0
y_true = torch.tensor(6.0)

# Forward：算预测值和 loss
y_pred = w * x_input + b
loss = (y_pred - y_true) ** 2

print("y_pred:", y_pred.item())
print("loss:", loss.item())

y_pred: 4.5
loss: 2.25


In [7]:
# 反向传播
loss.backward()

print("w.grad:", w.grad.item())
print("b.grad:", b.grad.item())

w.grad: -9.0
b.grad: -3.0


In [9]:
lr = 0.1

with torch.no_grad():   # 更新参数时不记录梯度
    w -= lr * w.grad
    b -= lr * b.grad

# 清空梯度，为下一步准备
w.grad.zero_()
b.grad.zero_()

print("w after update:", w.item())
print("b after update:", b.item())

# 再做一次 forward，看 loss 有没有变小
y_pred_new = w * x_input + b
loss_new = (y_pred_new - y_true) ** 2
print("new loss:", loss_new.item())

w after update: 2.4000000953674316
b after update: 0.30000001192092896
new loss: 2.2500014305114746


In [13]:
# 最优解应该是 w=2.0, b=0.0
# 用更小的学习率重新跑
w = torch.tensor(1.5, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

lr = 0.01   # 缩小10倍

for step in range(20):
    y_pred = w * 3.0 + b
    loss = (y_pred - y_true) ** 2

    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()

    if step % 5 == 0:
        print(f"step {step:2d} | loss: {loss.item():.4f} | w: {w.item():.4f} | b: {b.item():.4f}")

step  0 | loss: 2.2500 | w: 1.5900 | b: 0.0300
step  5 | loss: 0.2416 | w: 1.8320 | b: 0.1107
step 10 | loss: 0.0259 | w: 1.9113 | b: 0.1371
step 15 | loss: 0.0028 | w: 1.9373 | b: 0.1458


In [15]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# --- 模拟账单数据 ---
# 3个特征：金额、消费小时、是否周末(0或1)
# 3个类别：0=餐饮, 1=交通, 2=购物
X = torch.randn(100, 3)
y = torch.randint(0, 3, (100,))

# --- 切分训练集和验证集 ---
X_train, X_val = X[:80], X[80:]
y_train, y_val = y[:80], y[80:]

# --- 定义模型 ---
model = nn.Sequential(
    nn.Linear(3, 16),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(16, 3)
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- Training loop ---
for epoch in range(100):
    # 训练模式
    model.train()
    logits = model(X_train)
    loss = criterion(logits, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 每10个epoch，顺便看一下验证集
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val)
            val_loss = criterion(val_logits, y_val)
            val_acc = (val_logits.argmax(dim=1) == y_val).float().mean()

        print(f"Epoch {epoch:3d} | train loss: {loss.item():.4f} | val loss: {val_loss.item():.4f} | val acc: {val_acc.item():.2f}")

Epoch   0 | train loss: 1.1480 | val loss: 1.1310 | val acc: 0.20
Epoch  10 | train loss: 1.0637 | val loss: 1.1194 | val acc: 0.40
Epoch  20 | train loss: 1.0056 | val loss: 1.1710 | val acc: 0.45
Epoch  30 | train loss: 0.9853 | val loss: 1.2353 | val acc: 0.45
Epoch  40 | train loss: 0.9493 | val loss: 1.2600 | val acc: 0.45
Epoch  50 | train loss: 0.9664 | val loss: 1.2872 | val acc: 0.45
Epoch  60 | train loss: 0.9697 | val loss: 1.3097 | val acc: 0.45
Epoch  70 | train loss: 0.9399 | val loss: 1.3406 | val acc: 0.45
Epoch  80 | train loss: 0.9281 | val loss: 1.3623 | val acc: 0.45
Epoch  90 | train loss: 0.8872 | val loss: 1.3823 | val acc: 0.45


In [18]:
import pandas as pd

df = pd.read_csv("../data/personal_transactions.csv")  # 文件名以实际为准
print(df.shape)
print(df.head())
print(df.columns.tolist())

(806, 6)
         Date          Description   Amount Transaction Type  \
0  01/01/2018               Amazon    11.11            debit   
1  01/02/2018     Mortgage Payment  1247.44            debit   
2  01/02/2018      Thai Restaurant    24.22            debit   
3  01/03/2018  Credit Card Payment  2298.09           credit   
4  01/04/2018              Netflix    11.76            debit   

              Category   Account Name  
0             Shopping  Platinum Card  
1      Mortgage & Rent       Checking  
2          Restaurants    Silver Card  
3  Credit Card Payment  Platinum Card  
4        Movies & DVDs  Platinum Card  
['Date', 'Description', 'Amount', 'Transaction Type', 'Category', 'Account Name']


In [19]:
# 看看有哪些类别，各有多少条
print(df["Category"].value_counts())
print("\n只看支出:")
print(df["Transaction Type"].value_counts())

Category
Credit Card Payment       143
Groceries                 105
Restaurants                81
Utilities                  63
Shopping                   60
Gas & Fuel                 52
Paycheck                   46
Home Improvement           36
Coffee Shops               31
Alcohol & Bars             25
Mortgage & Rent            21
Music                      21
Mobile Phone               21
Internet                   21
Movies & DVDs              18
Auto Insurance             18
Fast Food                  16
Haircut                    13
Television                  8
Electronics & Software      4
Food & Dining               2
Entertainment               1
Name: count, dtype: int64

只看支出:
Transaction Type
debit     688
credit    118
Name: count, dtype: int64


In [20]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("../data/personal_transactions.csv")

# --- 1. 只保留支出，去掉非消费类别 ---
df = df[df["Transaction Type"] == "debit"].copy()
df = df[~df["Category"].isin(["Credit Card Payment", "Paycheck"])].copy()

# --- 2. 合并小类 ---
category_map = {
    "Restaurants":          "食饮",
    "Coffee Shops":         "食饮",
    "Alcohol & Bars":       "食饮",
    "Fast Food":            "食饮",
    "Food & Dining":        "食饮",
    "Groceries":            "食饮",
    "Shopping":             "购物",
    "Electronics & Software": "购物",
    "Home Improvement":     "购物",
    "Haircut":              "购物",
    "Gas & Fuel":           "出行",
    "Auto Insurance":       "出行",
    "Utilities":            "居家",
    "Mortgage & Rent":      "居家",
    "Internet":             "居家",
    "Mobile Phone":         "居家",
    "Movies & DVDs":        "娱乐",
    "Music":                "娱乐",
    "Television":           "娱乐",
    "Entertainment":        "娱乐",
}
df["label"] = df["Category"].map(category_map)
print(df["label"].value_counts())

label
食饮    260
居家    126
购物    113
出行     70
娱乐     48
Name: count, dtype: int64


In [21]:
# --- 3. 特征工程 ---
df["Date"] = pd.to_datetime(df["Date"])
df["hour"] = 12                                    # 数据没有时间，固定用中午占位
df["day_of_week"] = df["Date"].dt.dayofweek        # 0=周一, 6=周日
df["is_weekend"] = (df["day_of_week"] >= 5).astype(float)
df["amount_log"] = df["Amount"].apply(lambda x: __import__("math").log1p(x))  # 金额取log，压缩大数值

# Description 处理：提取关键词作为类别特征
# 先看看有哪些独特的 Description
print(f"独特商家数: {df['Description'].nunique()}")
print(df["Description"].value_counts().head(10))

独特商家数: 63
Description
Grocery Store       103
Amazon               59
Hardware Store       34
Starbucks            32
American Tavern      24
Brewing Company      23
Mortgage Payment     21
Gas Company          21
Spotify              21
Phone Company        21
Name: count, dtype: int64


In [22]:
import math

# --- 4. 编码 Description ---
# 63个商家，每个分配一个整数ID
desc_encoder = LabelEncoder()
df["desc_id"] = desc_encoder.fit_transform(df["Description"])

# --- 5. 编码 label ---
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

print("类别映射:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i} = {cls}")

# --- 6. 转成 tensor ---
# 数值特征：金额(log)、星期几、是否周末
X_num = torch.tensor(
    df[["amount_log", "day_of_week", "is_weekend"]].values,
    dtype=torch.float32
)
# 商家ID：整数，给 Embedding 层用
X_desc = torch.tensor(df["desc_id"].values, dtype=torch.long)
# 标签
y = torch.tensor(df["label_id"].values, dtype=torch.long)

print(f"\n数据量: {len(y)}")
print(f"X_num shape: {X_num.shape}")
print(f"X_desc shape: {X_desc.shape}")
print(f"y shape: {y.shape}")

# --- 7. 切分训练/验证集 ---
n = len(y)
n_train = int(n * 0.8)

# 先打乱顺序
idx = torch.randperm(n)
X_num  = X_num[idx]
X_desc = X_desc[idx]
y      = y[idx]

X_num_train,  X_num_val  = X_num[:n_train],  X_num[n_train:]
X_desc_train, X_desc_val = X_desc[:n_train], X_desc[n_train:]
y_train,      y_val      = y[:n_train],      y[n_train:]

print(f"\n训练集: {len(y_train)} 条")
print(f"验证集: {len(y_val)} 条")

类别映射:
  0 = 出行
  1 = 娱乐
  2 = 居家
  3 = 购物
  4 = 食饮

数据量: 617
X_num shape: torch.Size([617, 3])
X_desc shape: torch.Size([617])
y shape: torch.Size([617])

训练集: 493 条
验证集: 124 条


In [23]:
# --- 8. 定义模型 ---
class BillClassifier(nn.Module):
    def __init__(self, n_desc, embed_dim, n_num_features, n_classes):
        super().__init__()
        # Embedding层：把商家ID(整数) → 向量
        self.embedding = nn.Embedding(n_desc, embed_dim)
        
        # MLP：吃 embedding + 数值特征，输出分类
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim + n_num_features, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, n_classes)
        )
    
    def forward(self, x_desc, x_num):
        # x_desc: [batch_size]        → embedding后: [batch_size, embed_dim]
        emb = self.embedding(x_desc)
        # 拼接: [batch_size, embed_dim + n_num_features]
        x = torch.cat([emb, x_num], dim=1)
        return self.mlp(x)

model = BillClassifier(
    n_desc=63,        # 商家数量
    embed_dim=8,      # 每个商家用8维向量表示
    n_num_features=3, # amount_log, day_of_week, is_weekend
    n_classes=5       # 5个消费类别
)

print(model)
print(f"\n参数总量: {sum(p.numel() for p in model.parameters())}")

BillClassifier(
  (embedding): Embedding(63, 8)
  (mlp): Sequential(
    (0): Linear(in_features=11, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=5, bias=True)
  )
)

参数总量: 1053


In [24]:
# --- 9. 训练 ---
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    model.train()
    logits = model(X_desc_train, X_num_train)
    loss = criterion(logits, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_logits = model(X_desc_val, X_num_val)
            val_loss = criterion(val_logits, y_val)
            val_acc = (val_logits.argmax(dim=1) == y_val).float().mean()
        print(f"Epoch {epoch:3d} | train loss: {loss.item():.4f} | val loss: {val_loss.item():.4f} | val acc: {val_acc.item():.2f}")

Epoch   0 | train loss: 1.8176 | val loss: 1.6355 | val acc: 0.12
Epoch  10 | train loss: 1.0978 | val loss: 1.0357 | val acc: 0.59
Epoch  20 | train loss: 0.7277 | val loss: 0.5955 | val acc: 0.81
Epoch  30 | train loss: 0.4515 | val loss: 0.3215 | val acc: 0.96
Epoch  40 | train loss: 0.2920 | val loss: 0.1727 | val acc: 0.98
Epoch  50 | train loss: 0.1850 | val loss: 0.0918 | val acc: 0.98
Epoch  60 | train loss: 0.1164 | val loss: 0.0572 | val acc: 0.98
Epoch  70 | train loss: 0.0670 | val loss: 0.0414 | val acc: 0.99
Epoch  80 | train loss: 0.0533 | val loss: 0.0347 | val acc: 0.99
Epoch  90 | train loss: 0.0279 | val loss: 0.0328 | val acc: 0.99


In [25]:
# --- 10. 推理：看看模型实际在预测什么 ---
model.eval()
with torch.no_grad():
    # 取验证集前10条，对比预测和真实标签
    sample_logits = model(X_desc_val[:10], X_num_val[:10])
    sample_preds = sample_logits.argmax(dim=1)
    
    print(f"{'商家':<25} {'真实':>6} {'预测':>6} {'对否':>4}")
    print("-" * 45)
    for i in range(10):
        desc_id = X_desc_val[i].item()
        desc_name = desc_encoder.inverse_transform([desc_id])[0]
        true_label = label_encoder.inverse_transform([y_val[i].item()])[0]
        pred_label = label_encoder.inverse_transform([sample_preds[i].item()])[0]
        correct = "✓" if true_label == pred_label else "✗"
        print(f"{desc_name:<25} {true_label:>6} {pred_label:>6} {correct:>4}")

商家                            真实     预测   对否
---------------------------------------------
Amazon                        购物     购物    ✓
Spotify                       娱乐     娱乐    ✓
Amazon                        购物     购物    ✓
Grocery Store                 食饮     食饮    ✓
Gas Station                   出行     出行    ✓
Internet Service Provider     居家     居家    ✓
Brunch Restaurant             食饮     食饮    ✓
American Tavern               食饮     食饮    ✓
Starbucks                     食饮     食饮    ✓
State Farm                    出行     出行    ✓


In [26]:
# 保存模型权重
torch.save(model.state_dict(), "../data/bill_classifier.pth")
print("模型已保存")

模型已保存


In [ ]:
# 加载模型权重
model_loaded = BillClassifier(
    n_desc=63, embed_dim=8,
    n_num_features=3, n_classes=5
)
model_loaded.load_state_dict(torch.load("../data/bill_classifier.pth"))
model_loaded.eval()
print("模型已加载")